# M4.1: chunking strategy evaluation

This notebook compares Fixed, Recursive, and the production Structure-Aware Chunker. The dataset, embedding model, query encoding, cosine retrieval, and metrics are fixed; chunking strategy is the only intended variable. Ground truth uses source block IDs, never generated chunk IDs.

If a CUDA device-side assert has already occurred in this runtime, do not attempt recovery here: use **Runtime → Disconnect and delete runtime**, reopen the notebook, and use **Run all** from a fresh session.

In [1]:
# Run in a fresh Google Colab runtime. Replace the repository URL if using a fork.
import os
import subprocess
from pathlib import Path

REPOSITORY_URL = 'https://github.com/ozgemelteminan/prompt-generator-rag'  # Replace with your repository URL.
REPOSITORY_REF = 'main'  # Replace with the branch, tag, or commit you intend to benchmark.
repository = Path('prompt-generator-rag')
if not repository.exists():
    subprocess.run(['git', 'clone', REPOSITORY_URL], check=True)
else:
    subprocess.run(['git', '-C', str(repository), 'fetch', '--all', '--tags', '--prune'], check=True)
subprocess.run(['git', '-C', str(repository), 'checkout', REPOSITORY_REF], check=True)
current_branch = subprocess.run(['git', '-C', str(repository), 'branch', '--show-current'], check=True, capture_output=True, text=True).stdout.strip()
if current_branch:
    subprocess.run(['git', '-C', str(repository), 'pull', '--ff-only', 'origin', current_branch], check=True)
os.chdir(repository)
subprocess.run(['pip', 'install', '-q', '--upgrade', 'transformers==4.57.6', 'sentence-transformers==5.6.0'], check=True)
subprocess.run(['pip', 'install', '-q', '-e', 'packages/prompt-engine'], check=True)
subprocess.run(['pip', 'install', '-q', '-e', 'apps/api', '--no-deps'], check=True)

import torch
import transformers
import sentence_transformers

GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print('GPU:', GPU_NAME)
print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('sentence-transformers:', sentence_transformers.__version__)
assert transformers.__version__ == '4.57.6', (
    'M4.1 requires transformers==4.57.6. Restart/delete the Colab runtime and Run All from a fresh session.'
)
RUNTIME_METADATA = {
    'torchVersion': torch.__version__,
    'transformersVersion': transformers.__version__,
    'sentenceTransformersVersion': sentence_transformers.__version__,
    'cudaDevice': GPU_NAME,
}

# Explicitly support both repository evaluation modules and the production app package.
import sys
repository_root = Path.cwd().resolve()
api_root = repository_root / 'apps' / 'api'
for import_root in (repository_root, api_root):
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))

stale_modules = [name for name in sys.modules if name == 'app' or name.startswith('app.') or name == 'evals' or name.startswith('evals.')]
if stale_modules:
    raise RuntimeError('Stale evaluation modules are already loaded. Restart the Colab runtime, then use Run All.')

GPU: CPU
torch: 2.11.0+cpu
transformers: 4.57.6
sentence-transformers: 5.6.0


In [2]:
import inspect
from pathlib import Path
from evals.src.dataset import load_dataset
from evals.src.chunking_eval import (
    SentenceTransformerEmbedder, fixed_size_chunks, recursive_chunks,
    production_structure_aware_chunks, run_comparison, save_results,
)
from app.document_processing.models import ChunkingConfig

if 'trust_remote_code' not in inspect.signature(SentenceTransformerEmbedder.__init__).parameters:
    raise RuntimeError('Stale SentenceTransformerEmbedder detected. Restart the runtime and use Run All.')

ROOT = Path.cwd()
dataset = load_dataset(ROOT / 'evals/datasets/chunking_eval_v1.json')
MODEL = 'Alibaba-NLP/gte-multilingual-base'
embedder = SentenceTransformerEmbedder(model_name=MODEL, trust_remote_code=True)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/55.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

Some weights of the model checkpoint at Alibaba-NLP/gte-multilingual-base were not used when initializing NewModel: ['classifier.bias', 'classifier.weight']
- This IS expected if you are initializing NewModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing NewModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [3]:
production_config = ChunkingConfig(target_tokens=350, max_tokens=500, overlap_tokens=40)
strategies = {
    'fixed': tuple(chunk for document in dataset.documents for chunk in fixed_size_chunks(document, max_tokens=500, overlap_tokens=40)),
    'recursive': tuple(chunk for document in dataset.documents for chunk in recursive_chunks(document, target_tokens=350, max_tokens=500)),
    # Calls apps/api/app/document_processing/chunking.py; it is not copied here.
    'production_structure_aware': tuple(chunk for document in dataset.documents for chunk in production_structure_aware_chunks(document, config=production_config)),
}
configurations = {
    'fixed': {'max_tokens': 500, 'overlap_tokens': 40},
    'recursive': {'target_tokens': 350, 'max_tokens': 500},
    'production_structure_aware': {'target_tokens': 350, 'max_tokens': 500, 'overlap_tokens': 40},
}
results = run_comparison(dataset, embedder=embedder, strategies=strategies)

In [4]:
import pandas as pd

comparison = pd.DataFrame([{'Chunker': result.name, 'Avg Tokens': result.chunk_statistics['mean_tokens'], 'Recall@5': result.overall['recall_at_5'], 'Recall@10': result.overall['recall_at_10'], 'MRR': result.overall['mrr'], 'nDCG@10': result.overall['ndcg_at_10'], 'HitRate@5': result.overall['hit_rate_at_5']} for result in results])
display(comparison)
for metric, label in [('Recall@10', 'best Recall@10'), ('MRR', 'best MRR'), ('nDCG@10', 'best nDCG@10')]:
    winners = comparison.loc[comparison[metric] == comparison[metric].max(), 'Chunker'].tolist()
    print(f'{label}: {winners}')
for result in results:
    print(f'\n{result.name}: category breakdown')
    display(pd.DataFrame(result.by_category).T)
    print(f'{result.name}: language breakdown')
    display(pd.DataFrame(result.by_language).T)

save_results(results, dataset_version=dataset.version, embedder=embedder, output_dir=ROOT / 'evals/results/chunking', chunker_configurations=configurations, runtime_metadata=RUNTIME_METADATA)

,Chunker,Avg Tokens,Recall@5,Recall@10,MRR,nDCG@10,HitRate@5
0,fixed,64.666667,1.00000,1.0,0.844048,0.779164,1.00000
1,recursive,64.666667,0.97619,1.0,0.831349,0.769929,0.97619
2,production_structure_aware,32.333333,1.00000,1.0,0.898810,0.817803,1.00000


best Recall@10: ['fixed', 'recursive', 'production_structure_aware']
best MRR: ['production_structure_aware']
best nDCG@10: ['production_structure_aware']

fixed: category breakdown


,recall_at_5,recall_at_10,hit_rate_at_5,mrr,ndcg_at_10
cross_paragraph,1.0,1.0,1.0,0.763889,0.503872
factual,1.0,1.0,1.0,0.958333,0.969244
heading_dependent,1.0,1.0,1.0,0.866667,0.550489
morphology_heavy,1.0,1.0,1.0,0.722222,0.729168
paraphrase,1.0,1.0,1.0,0.722222,0.793643
terminology_mismatch,1.0,1.0,1.0,0.916667,0.938488


fixed: language breakdown


,recall_at_5,recall_at_10,hit_rate_at_5,mrr,ndcg_at_10
en,1.0,1.0,1.0,0.799206,0.760837
tr,1.0,1.0,1.0,0.888889,0.797491


...

,recall_at_5,recall_at_10,hit_rate_at_5,mrr,ndcg_at_10
en,0.952381,1.0,0.952381,0.773810,0.742367
tr,1.000000,1.0,1.000000,0.888889,0.797491



production_structure_aware: category breakdown


,recall_at_5,recall_at_10,hit_rate_at_5,mrr,ndcg_at_10
cross_paragraph,1.0,1.0,1.0,0.916667,0.639907
factual,1.0,1.0,1.0,0.958333,0.969244
heading_dependent,1.0,1.0,1.0,1.000000,0.613147
morphology_heavy,1.0,1.0,1.0,0.916667,0.874013
paraphrase,1.0,1.0,1.0,0.708333,0.782089
terminology_mismatch,1.0,1.0,1.0,0.833333,0.876977


production_structure_aware: language breakdown


,recall_at_5,recall_at_10,hit_rate_at_5,mrr,ndcg_at_10
en,1.0,1.0,1.0,0.904762,0.819172
tr,1.0,1.0,1.0,0.892857,0.816435


## Reporting rule

Report the best Recall@10, MRR, and nDCG@10 independently. Recommend a strategy only when it wins the quality metric relevant to the use case without an unacceptable chunk-size or truncation tradeoff; otherwise report the tradeoff. Do not create a composite score.